# Solving the HEOM steady state without blowing up your memory
### A hands-on tutorial on the iterative steady-state solver of Kaspar & Thoss (J. Phys. Chem. A 2021, 125, 5190)

The **hierarchical equations of motion (HEOM)** are a numerically exact tool for open quantum systems — for example a molecule sandwiched between two metal electrodes. The price of exactness is the number of **auxiliary density operators (ADOs)**, which explodes with low temperature, strong coupling, and system size.

If you only want the **steady state** (e.g. the current through a molecular junction), you have two classic options:

| Strategy | Problem |
|---|---|
| Propagate in time until things stop changing | Can take very long to relax |
| Assemble one giant sparse matrix and solve $\mathcal{L}\rho=0$ directly (LU) | Memory explodes through *fill-in* |

Kaspar & Thoss propose a third way: **never build the big matrix**. Treat the whole hierarchy as a set of *coupled matrix equations* whose unknowns are the small $d\times d$ ADOs, and solve it with a Krylov method (CGNE) in which every operation is just a $d\times d$ matrix–matrix product. Two extra ingredients make it efficient:

1. a cheap **preconditioner** that is diagonal in the eigenbasis of $H_S$ (a Sylvester equation that needs no factorization), and
2. a **hierarchy truncation** that eliminates the top tier algebraically, using the same Sylvester solve.

**What this notebook does.** It is a compact NumPy re-implementation written for teaching — *not* the authors' code. It handles a spinless tight-binding chain between two Lorentzian leads, where the exact answer is known (Landauer formula), so every step can be tested. Expect the numbers to illustrate the ideas, not to reproduce the paper's timings (this is pure Python on one CPU core).

## 1. The big idea: the HEOM as coupled matrix equations

In the steady state the HEOM read $0=\mathcal{L}_n(\tilde\rho^{(0)},\tilde\rho^{(1)},\dots)$ for every ADO $\tilde\rho^{(n)}_{j_n\dots j_1}$:

$$
0=-i[\tilde H_S,\tilde\rho^{(n)}]-\sum_{k}\gamma_{j_k}\tilde\rho^{(n)}
-i\sum_j \tilde{\mathcal A}_{\bar j}\,\tilde\rho^{(n+1)}_{j\dots}
-i\sum_k(\pm)\,\tilde{\mathcal C}_{j_k}\,\tilde\rho^{(n-1)}_{\dots/j_k}
$$

Stack all ADOs into one object $X=(\tilde\rho^{(0)},\tilde\rho^{(1)},\dots)$ and call the right-hand side $\mathcal{A}(X)$. Then the steady state is $\mathcal A(X)=0$ with $\mathrm{Tr}\,\tilde\rho^{(0)}=1$.

* A normal solver would flatten $X$ into a vector of length $N_{\rm ADO}\,d^2$ and build an $(N_{\rm ADO}d^2)^2$ matrix.
* Instead we only ever apply $\mathcal A$ to $X$: every term has the form $L\,X_s\,R$ with small $d\times d$ matrices — **matrix-free**.

Because CGNE also needs the **adjoint** $\mathcal A^\dagger$, and a term $X\mapsto cLXR$ has the adjoint $Y\mapsto \bar c\,L^\dagger YR^\dagger$, we can store the operator as a plain *list of terms* and get both for free. That is exactly how the code below is organised.

> **Implementation details the paper does not spell out** (I had to choose them): the steady-state equations are singular (any multiple of a solution is a solution), so I replace the equation for $\tilde\rho^{(0)}$ by $\mathcal A_0(X)+\mathrm{Tr}(\rho^{(0)})\,\mathbb 1/d=\mathbb 1/d$, which pins the trace to 1. And fermionic ADOs anticommute, so I store each ADO once, with *sorted* indices, and track the permutation sign.

In [ ]:
import itertools, time
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt

trapz = getattr(np, "trapezoid", None) or np.trapz   # works on old & new NumPy
np.set_printoptions(precision=6, suppress=True)

## 2. The bath: exponential expansion of the correlation functions

HEOM needs the lead correlation functions as sums of exponentials, $C^\sigma(t)\approx\sum_l\eta_l e^{-\gamma_l t}$. For a Lorentzian level-width function $\Gamma(\epsilon)=\Gamma W^2/((\epsilon-\mu)^2+W^2)$ the poles come from

* one pole of the Lorentzian itself, and
* poles of the Fermi function. We use a **Padé** expansion, $f(y)\approx\tfrac12-\sum_{p=1}^{N_P}\frac{2\kappa_p\,y}{y^2+\zeta_p^2}$ (the Matsubara expansion would be $\kappa_p=1,\ \zeta_p=(2p-1)\pi$).

The paper's Padé coefficients come from the literature; here I get the poles from the eigenvalues of a tridiagonal matrix and the residues by matching the Taylor series of $\tfrac12\tanh(y/2)$. That is accurate for the handful of poles we use (it becomes ill-conditioned beyond $N_P\approx 8$).

In [ ]:
def pade_fermi(N):
    """[N-1/N] Pade: f(y)=1/(e^y+1) ~ 1/2 - sum_p 2 kappa_p y/(y^2+zeta_p^2).
    Poles from the eigenvalues of a tridiagonal matrix; residues from matching the
    Taylor expansion of tanh(y/2)/2 (fine for N <~ 8 in double precision)."""
    from math import factorial
    from scipy.special import bernoulli
    M = 2*N
    A = np.zeros((M, M))
    for j in range(M):
        for k in range(M):
            if abs(j - k) == 1:
                A[j, k] = 1.0 / np.sqrt((2*(j+1)-1) * (2*(k+1)-1))
    w = np.linalg.eigvalsh(A)
    zeta = np.sort(2.0 / w[w > 1e-12])
    B = bernoulli(2*N + 2)
    c = [0.5*(2**(2*n)*(2**(2*n)-1)*B[2*n]/factorial(2*n))*0.5**(2*n-1) for n in range(1, N+1)]
    Mx = np.array([[2*(-1)**m*z**(-2*(m+1)) for z in zeta] for m in range(N)])
    kappa = np.linalg.solve(Mx, np.array(c))
    return kappa, zeta

def fermi_pade(y, kappa, zeta):
    y = np.asarray(y, dtype=complex)
    return 0.5 - sum(2*k*y/(y**2 + z**2) for k, z in zip(kappa, zeta))

In [ ]:
# --- test: Pade approximation of the Fermi function
y = np.linspace(-20, 20, 2001)
exact = 1/(np.exp(y) + 1)
for Np in (1, 2, 3, 4, 6):
    kap, zet = pade_fermi(Np)
    print(f"N_P={Np}:  max |f_Pade - f| on [-20,20] = {np.max(abs(fermi_pade(y, kap, zet) - exact)):.1e}")

In [ ]:
# ------------------------------------------------- bath correlation expansion
def lorentzian_bath_poles(Gamma, W, mu, T, Np):
    """Exponential expansion C^sigma(t) = sum_l eta_l exp(-gamma_l t), t>0, for
    Gamma(e)=Gamma W^2/((e-mu)^2+W^2). Returns {sigma: (eta, gamma)}."""
    kap, zet = pade_fermi(Np)
    out = {}
    for s in (+1, -1):
        # Lorentzian pole at x = s*i*W,  f^+ = f, f^- = 1 - f
        f = fermi_pade(s*1j*W/T, kap, zet)
        fs = f if s == +1 else 1 - f
        eta = [0.5*Gamma*W*fs]
        gam = [W + s*(-1j)*mu]                       # W - i s mu
        for k, z in zip(kap, zet):                   # Pade poles
            eta.append(-1j*k*T*Gamma*W**2/(W**2 - (z*T)**2))
            gam.append(z*T - 1j*s*mu)
        out[s] = (np.array(eta, dtype=complex), np.array(gam, dtype=complex))
    return out

In [ ]:
# --- test: does sum_l eta_l exp(-gamma_l t) reproduce C^sigma(t) = (1/2pi) int dE Gamma(E) f^sigma(E) exp(i sigma E t) ?
Gam, W, T, mu = 1.0, 10.0, 1.0, 0.7
poles = lorentzian_bath_poles(Gam, W, mu, T, Np=5)
E = np.linspace(-3000, 3000, 2_000_001)
G = Gam*W**2/((E-mu)**2 + W**2)
f = 0.5*(1 - np.tanh((E-mu)/(2*T)))
for s in (+1, -1):
    fs = f if s == +1 else 1 - f
    for t in (0.1, 0.5, 2.0):
        ex = trapz(G*fs*np.exp(1j*s*E*t), E)/(2*np.pi)
        ap = np.sum(poles[s][0]*np.exp(-poles[s][1]*t))
        print(f"sigma={s:+d}  t={t:3.1f}   exact={ex:.5f}   expansion={ap:.5f}   |diff|={abs(ex-ap):.1e}")

## 3. The model: a chain of electronic sites between two leads

We use $N_{\rm el}$ spinless sites with on-site energy $\epsilon_0$ and nearest-neighbour hopping $t$; the left (right) lead couples to the first (last) site. Everything is built in Fock space with a Jordan–Wigner transformation (dimension $d=2^{N_{\rm el}}$) and then rotated into the **eigenbasis of $H_S$** — in that basis $-i[H_S,\cdot]$ is a simple element-wise factor, which is what makes the preconditioner cheap.

Because the model is non-interacting, we have an exact reference: the **Landauer formula** with the Lorentzian self-energy $\Sigma_K(\epsilon)=\frac{\Gamma W/2}{\epsilon-\mu_K+iW}$. A hierarchy of tier 2 is exact for this model.

In [ ]:
# ------------------------------------------------------ model: chain of sites
def chain_system(Nel, eps0, t):
    """Spinless tight-binding chain in Fock space (Jordan-Wigner); returns eigenbasis ops."""
    dim = 2**Nel
    a = np.array([[0, 1], [0, 0]], float)         # |0>,|1> basis, annihilation
    Z = np.diag([1., -1.]); I2 = np.eye(2)
    def kron_list(L):
        out = np.eye(1)
        for m in L: out = np.kron(out, m)
        return out
    c = [kron_list([Z]*i + [a] + [I2]*(Nel-i-1)) for i in range(Nel)]
    H = sum(eps0*ci.T @ ci for ci in c)
    for i in range(Nel-1):
        H = H + t*(c[i].T @ c[i+1] + c[i+1].T @ c[i])
    E, U = np.linalg.eigh(H)
    ct = [U.T @ ci @ U for ci in c]
    return E, ct, (eps0*np.eye(Nel) + t*(np.eye(Nel, k=1) + np.eye(Nel, k=-1)))

def landauer_current(h1, Gamma, W, muL, muR, T, nE=60001, Emax=None):
    """Landauer current for the chain with Lorentzian level-width functions (hbar=e=1)."""
    Nel = h1.shape[0]; Emax = Emax or 60*W
    e = np.linspace(-Emax, Emax, nE)
    f = lambda x, mu: 0.5*(1 - np.tanh((x-mu)/(2*T)))
    sL = 0.5*Gamma*W/(e - muL + 1j*W); sR = 0.5*Gamma*W/(e - muR + 1j*W)
    M = (e[:, None, None]*np.eye(Nel) - h1).astype(complex)
    M[:, 0, 0] -= sL; M[:, -1, -1] -= sR
    G1N = np.linalg.inv(M)[:, 0, -1]
    gL = Gamma*W**2/((e-muL)**2 + W**2); gR = Gamma*W**2/((e-muR)**2 + W**2)
    return np.trapezoid(gL*gR*abs(G1N)**2*(f(e, muL) - f(e, muR)), e)/(2*np.pi)

## 4. Building the HEOM operator

`HEOM` enumerates all ADOs up to the maximal tier (index sets `itertools.combinations(range(M), n)`, because fermionic ADOs are antisymmetric under index exchange) and generates the list of terms. The class provides

* `A(X)` and `Aadj(Y)` — the matrix-free operator and its adjoint,
* `Pinv` — the **preconditioner** (Sec. 3.2 of the paper): the inverse of the diagonal part $\mathcal P(X)_{ab}=\big(-i(E_a-E_b)-\sum_k\gamma_{j_k}\big)X_{ab}$. In the eigenbasis this Sylvester equation is solved by *element-wise division*, no factorization needed. Tier 0 is left unpreconditioned, because $-i[H_S,\cdot]$ is singular on the diagonal.
* `truncate=True` — **hierarchy truncation** (Sec. 3.3): the top tier $N_T$ is never stored. Setting its time derivative to zero gives a Sylvester equation $\mathcal P(\rho^{(N_T)})=-(\dots)\rho^{(N_T-1)}$, solved by the same element-wise division, and the result is inserted into tier $N_T-1$. In the code these are the "two-stage" terms $L_2[(L_1XR_1)\circ W]R_2$.
* `assemble()` — builds the explicit sparse matrix, **only** to provide a direct-solver reference.

In [ ]:
# ------------------------------------------------------------- HEOM builder
class HEOM:
    """Steady-state HEOM as a set of coupled matrix equations (in the eigenbasis of H_S).
    X = array (n_ado, d, d). Operator applied matrix-free; adjoint is also matrix-free."""
    def __init__(self, E, dops, baths, NT, truncate=False):
        # E: eigenvalues of H_S; dops[K]: annihilation operator (eigenbasis) coupled to lead K
        # baths[K] = {sigma: (eta, gamma)}
        self.E, self.d = np.asarray(E), len(E)
        d = self.d
        self.NT = NT
        self.truncate = truncate
        # multi-index j = (K, sigma, l) -> integer
        J = []
        for K, b in enumerate(baths):
            for s in (+1, -1):
                for l in range(len(b[s][0])):
                    J.append((K, s, l))
        self.M = M = len(J)
        eta = np.array([baths[K][s][0][l] for K, s, l in J])
        gam = np.array([baths[K][s][1][l] for K, s, l in J])
        jbar = [J.index((K, -s, l)) for K, s, l in J]
        self.J, self.eta, self.gam, self.jbar = J, eta, gam, jbar
        op = lambda j: (dops[J[j][0]].conj().T if J[j][1] == +1 else dops[J[j][0]])  # d^sigma
        Neff = NT - 1 if truncate else NT
        self.Neff = Neff
        self.ados = [c for n in range(Neff+1) for c in itertools.combinations(range(M), n)]
        self.idx = {c: i for i, c in enumerate(self.ados)}
        self.n_ado = len(self.ados)
        Ediff = self.E[:, None] - self.E[None, :]
        # diagonal part  -i[H,.] - sum gamma
        D = np.zeros((self.n_ado, d, d), complex)
        for c, i in self.idx.items():
            D[i] = -1j*Ediff - gam[list(c)].sum()
        self.D = D
        # --- helper: C_j and A_jbar as lists of (coef, L, R) acting on operand of tier m
        def C_terms(j, m):
            return [(eta[j], op(j), None), (-(-1)**m * np.conj(eta[jbar[j]]), None, op(j))]
        def A_terms(j, m):
            ob = op(jbar[j])
            return [(1.0, ob, None), ((-1.0)**m, None, ob)]
        dag = lambda A: None if A is None else A.conj().T
        terms = []
        def add(t, s, c, L1, R1, W, L2, R2):
            terms.append((t, s, c, L1, R1, W, L2, R2,
                          dag(L1), dag(R1), None if W is None else W.conj(), dag(L2), dag(R2)))
        for c, t in self.idx.items():
            n = len(c)
            # tier-down:  -i (-1)^{n-r} C_{j_r} rho_{c \ j_r}
            for r, jr in enumerate(c):
                s_idx = self.idx[c[:r] + c[r+1:]]
                sgn = (-1)**r
                for (cf, L, R) in C_terms(jr, n-1):
                    add(t, s_idx, -1j*sgn*cf, None, None, None, L, R)
            # tier-up
            if n < Neff:
                for j in range(M):
                    if j in c: continue
                    Jc = tuple(sorted(c + (j,)))
                    p = (-1)**sum(1 for i in c if i < j)
                    for (cf, L, R) in A_terms(j, n+1):
                        add(t, self.idx[Jc], -1j*p*cf, None, None, None, L, R)
            elif truncate and n == Neff:   # eliminate tier NT algebraically (Sec 3.3)
                for j in range(M):
                    if j in c: continue
                    Jc = tuple(sorted(c + (j,)))
                    p = (-1)**sum(1 for i in c if i < j)
                    Winv = 1.0 / (-1j*Ediff - gam[list(Jc)].sum())
                    for r, jr in enumerate(Jc):
                        s_idx = self.idx[Jc[:r] + Jc[r+1:]]
                        sgn = (-1)**r
                        for (c1, L1, R1) in C_terms(jr, NT-1):
                            for (c2, L2, R2) in A_terms(j, NT):
                                add(t, s_idx, p*sgn*c1*c2, L1, R1, Winv, L2, R2)
        self.terms = terms
        # preconditioner: elementwise inverse of the diagonal part D (Sec. 3.2).
        # Tier 0 is left unpreconditioned (-i[H,.] is singular on the diagonal).
        self.Pinv = np.ones_like(D)
        self.Pinv[1:] = 1.0 / D[1:]
        self.Id = np.eye(d)

    # ---- the linear operator A(X) with normalisation row Tr rho0 = 1  ----
    def A(self, X):
        out = self.D * X
        for (t, s, c, L1, R1, W, L2, R2, *_) in self.terms:
            Z = X[s]
            if L1 is not None: Z = L1 @ Z
            if R1 is not None: Z = Z @ R1
            if W is not None: Z = Z * W
            if L2 is not None: Z = L2 @ Z
            if R2 is not None: Z = Z @ R2
            out[t] += c * Z
        out[0] += np.trace(X[0]) * self.Id / self.d     # normalisation (fixes singular null-space)
        return out

    def Aadj(self, Y):
        out = self.D.conj() * Y
        for (t, s, c, L1, R1, W, L2, R2, L1d, R1d, Wc, L2d, R2d) in self.terms:
            Z = Y[t]
            if L2d is not None: Z = L2d @ Z
            if R2d is not None: Z = Z @ R2d
            if Wc is not None: Z = Z * Wc
            if L1d is not None: Z = L1d @ Z
            if R1d is not None: Z = Z @ R1d
            out[s] += np.conj(c) * Z
        out[0] += np.trace(Y[0]) * self.Id / self.d
        return out

    def rhs(self):
        b = np.zeros((self.n_ado, self.d, self.d), complex)
        b[0] = self.Id / self.d
        return b

    # ---- current into the molecule from lead K:  I_K = 2 Re[ i sum_l Tr( d_K rho_{(K,+,l)} ) ] ----
    def current(self, X, dops, K):
        tot = 0
        for j, (k, s, l) in enumerate(self.J):
            if k == K and s == +1:
                tot += 1j*np.trace(dops[K] @ X[self.idx[(j,)]])
        return 2*tot.real

    # ---- sparse assembly for the direct solver reference ----
    def assemble(self):
        d2, n = self.d**2, self.n_ado
        I = sp.identity(self.d, format='csr')
        kr = lambda L, R: sp.kron(sp.csr_matrix(I if L is None else L),
                                  sp.csr_matrix(I if R is None else R.T), format='csr')
        rows, cols, vals = [], [], []
        def put(t, s, B):
            B = sp.coo_matrix(B)
            rows.append(B.row + t*d2); cols.append(B.col + s*d2); vals.append(B.data)
        for i in range(n):
            put(i, i, sp.diags(self.D[i].ravel()))
        for (t, s, c, L1, R1, W, L2, R2, *_) in self.terms:
            B = kr(L2, R2)
            if W is not None:
                B = B @ sp.diags(W.ravel()) @ kr(L1, R1)
            put(t, s, c*B)
        vi = self.Id.ravel()
        put(0, 0, sp.csr_matrix(np.outer(vi, vi)/self.d))
        A = sp.coo_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))),
                          shape=(n*d2, n*d2)).tocsc()
        return A

## 5. The solvers

* `solve_direct` — assemble the sparse matrix and LU-factorise it (the "DS" reference; I use SciPy's sparse LU, which is much friendlier than a dense LU).
* `solve_cgne` — the iterative solver. CGNE is the conjugate-gradient method applied to the normal equations of the second kind; the paper picked it after comparing BiCGSTAB, GPBiCG, CGS, LSQR, LSMR, and others. The residual is the Frobenius norm $\|\mathcal A(X)-b\|_F$ (stopping criterion in the paper: $10^{-12}$).

The paper applies the preconditioner from the right ($\mathcal A\mathcal P^{-1}Y=b$, $X=\mathcal P^{-1}Y$). I also include a **left** variant ($\mathcal P^{-1}\mathcal A X=\mathcal P^{-1}b$), because in this toy model it behaves better — see test 4.

In [ ]:
# ------------------------------------------------------------------ solvers
def solve_direct(h):
    A = h.assemble()
    b = h.rhs().ravel()
    x = spla.spsolve(A, b)
    return x.reshape(h.n_ado, h.d, h.d)

def solve_cgne(h, tol=1e-10, maxit=20000, precond='left'):
    """Matrix-free CGNE for the coupled matrix equations A(X) = b.
    precond: None | 'left' (P^-1 A X = P^-1 b) | 'right' (A P^-1 Y = b, X = P^-1 Y)."""
    b = h.rhs(); Pi = h.Pinv
    if precond == 'left':
        M, Mt, b = (lambda X: Pi*h.A(X)), (lambda R: h.Aadj(Pi.conj()*R)), Pi*b
        post = lambda Y: Y
    elif precond == 'right':
        M, Mt = (lambda Y: h.A(Pi*Y)), (lambda R: Pi.conj()*h.Aadj(R))
        post = lambda Y: Pi*Y
    else:
        M, Mt, post = h.A, h.Aadj, (lambda Y: Y)
    Y = np.zeros_like(b)
    r = b.copy(); p = Mt(r)
    rr = np.vdot(r, r).real
    hist = [np.sqrt(rr)]
    while hist[-1] > tol and len(hist) <= maxit:
        q = M(p)
        alpha = rr / np.vdot(p, p).real
        Y += alpha * p
        r -= alpha * q
        rr_new = np.vdot(r, r).real
        p = Mt(r) + (rr_new / rr) * p
        rr = rr_new
        hist.append(np.sqrt(rr))
    return post(Y), len(hist)-1, np.array(hist)

## 6. Tests

All tests use $\Gamma=1$ (our unit of energy), $W=10$, $T=1$, $\epsilon_0=0.3$, $t=1$, and $N_P=3$ Padé poles per lead and spin-direction, unless stated otherwise. Hierarchy tier $N_T=2$.

In [ ]:
Gamma, W, T, eps0, t_hop = 1.0, 10.0, 1.0, 0.3, 1.0

def make_heom(Nel, V, Np=3, NT=2, truncate=True):
    E, ct, h1 = chain_system(Nel, eps0, t_hop)
    muL, muR = V/2, -V/2
    baths = [lorentzian_bath_poles(Gamma, W, muL, T, Np),
             lorentzian_bath_poles(Gamma, W, muR, T, Np)]
    dops = [ct[0], ct[-1]]
    h = HEOM(E, dops, baths, NT, truncate)
    land = landauer_current(h1, Gamma, W, muL, muR, T)
    return h, dops, land

### Test 1 — Is the operator right, and do all three solvers agree with Landauer?

First check that the matrix-free adjoint really is the adjoint, $\langle Y,\mathcal A X\rangle=\langle\mathcal A^\dagger Y,X\rangle$; then solve the two-site chain at bias $V=2$ with the direct solver and with CGNE.

In [ ]:
h, dops, land = make_heom(Nel=2, V=2.0)
rng = np.random.default_rng(1)
X = rng.normal(size=h.D.shape) + 1j*rng.normal(size=h.D.shape)
Y = rng.normal(size=h.D.shape) + 1j*rng.normal(size=h.D.shape)
print("adjoint test  |<Y,AX> - <A^dag Y,X>| =", abs(np.vdot(Y, h.A(X)) - np.vdot(h.Aadj(Y), X)))

print(f"\nLandauer reference current           : {land:.8f}")
for trunc in (False, True):
    h, dops, _ = make_heom(2, 2.0, truncate=trunc)
    t0 = time.time(); Xd = solve_direct(h);               td = time.time() - t0
    t0 = time.time(); Xi, it, hist = solve_cgne(h);       ti = time.time() - t0
    print(f"truncation={trunc!s:5}  N_ADO={h.n_ado:4d}"
          f" | direct: I={h.current(Xd, dops, 0):.8f} ({td:.2f}s)"
          f" | CGNE: I={h.current(Xi, dops, 0):.8f} ({ti:.2f}s, {it} iterations)")
    print(f"{'':27s}trace(rho0)={np.trace(Xi[0]).real:.12f}   I_L + I_R = {h.current(Xi, dops, 0) + h.current(Xi, dops, 1):.1e}")

Both solvers should agree with each other to ~$10^{-11}$ and with Landauer to the accuracy of the $N_P=3$ Padé expansion. The truncated hierarchy uses 17 ADOs instead of 137 and gives the *same* answer, as it must for a non-interacting model where tier 2 is exact.

### Test 2 — A full current–voltage curve

In [ ]:
Vs = np.linspace(0, 5, 11)
I_heom, I_land, its = [], [], []
for V in Vs:
    h, dops, land = make_heom(2, V)
    X, it, _ = solve_cgne(h)
    I_heom.append(h.current(X, dops, 0)); I_land.append(land); its.append(it)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].plot(Vs, I_land, "-", label="Landauer (exact)")
ax[0].plot(Vs, I_heom, "o", mfc="none", label="HEOM, iterative steady state")
ax[0].set_xlabel("bias voltage $\\Phi$ [$\\Gamma$]"); ax[0].set_ylabel("current [$\\Gamma$]"); ax[0].legend()
ax[1].semilogy(Vs, np.abs(np.array(I_heom) - np.array(I_land)) + 1e-16, "s-")
ax[1].set_xlabel("bias voltage $\\Phi$ [$\\Gamma$]"); ax[1].set_ylabel("|I_HEOM - I_Landauer|")
ax[1].set_title("residual error (Pade truncation)")
plt.tight_layout(); plt.show()
print("CGNE iterations per bias point:", its)

### Test 3 — Convergence in the number of Padé poles

The remaining discrepancy in Test 2 is not a solver error but the finite number of Padé poles $N_P$. With enough poles, HEOM and Landauer agree to many digits — the “numerically exact” promise of HEOM. (More poles mean more ADOs, which is why an efficient solver matters.)

In [ ]:
print(" N_P   N_ADO (truncated)   I_HEOM        |I_HEOM - I_Landauer|")
for Np in (1, 2, 3, 4, 5, 6):
    h, dops, land = make_heom(2, 2.0, Np=Np)
    X, it, _ = solve_cgne(h)
    I = h.current(X, dops, 0)
    print(f"  {Np}   {h.n_ado:8d}           {I:.8f}    {abs(I-land):.1e}")

### Test 4 — Do the preconditioner and the truncation help?

This mirrors Table 2 of the paper (variants IT/PT, IT/T, IT/P, IT) for the two-site chain. We count CGNE iterations needed to reach a residual of $10^{-10}$, for the full and truncated hierarchy, with no preconditioner, the paper's right preconditioner, and a left preconditioner.

In [ ]:
print(f"{'hierarchy':12s}{'N_ADO':>7s}   {'precond.':9s}{'iterations':>11s}{'time [s]':>10s}")
curves = {}
for trunc in (False, True):
    h, dops, land = make_heom(2, 2.0, truncate=trunc)
    for pc in (None, "right", "left"):
        t0 = time.time(); X, it, hist = solve_cgne(h, precond=pc, maxit=20000); dt = time.time() - t0
        curves[(trunc, pc)] = hist
        print(f"{'truncated' if trunc else 'full':12s}{h.n_ado:7d}   {str(pc):9s}{it:11d}{dt:10.2f}")

plt.figure(figsize=(6.5, 3.8))
for (trunc, pc), hist in curves.items():
    plt.semilogy(hist, ("-" if trunc else "--"), label=f"{'trunc.' if trunc else 'full'}, {pc}")
plt.xlabel("CGNE iteration"); plt.ylabel("residual $\\|\\mathcal{A}(X)-b\\|_F$"); plt.legend(fontsize=8); plt.tight_layout(); plt.show()

**What to expect here.** Truncation is the clear winner: fewer ADOs *and* fewer iterations. The preconditioner's effect is regime dependent. In this toy model (moderate temperature, modest energy spread, my tier-0 treatment) the paper's *right* preconditioner is not a clear win and can even hurt on the full hierarchy, whereas the *left* variant roughly halves the iteration count. The paper reports a much larger preconditioning gain for its interacting model (Table 2), so treat this as an invitation to experiment rather than a contradiction.

### Test 5 — The point of it all: memory and scaling with system size

The direct solver has to store the LU factors of a matrix of dimension $N_{\rm ADO}d^2$; the iterative solver only stores the ADOs themselves ($N_{\rm ADO}\cdot d^2$ complex numbers) and works with $d\times d$ products. We lengthen the chain ($d=2^{N_{\rm el}}$) and compare, for the **full** hierarchy and for the truncated one.

In [ ]:
rows = []
for Nel in (1, 2, 3):
    for trunc in (True, False):
        h, dops, land = make_heom(Nel, 2.0, truncate=trunc)
        t0 = time.time(); A = h.assemble(); lu = spla.splu(A); Xd = lu.solve(h.rhs().ravel()).reshape(h.n_ado, h.d, h.d); td = time.time() - t0
        mem_lu = (lu.L.nnz + lu.U.nnz)*16/1e6
        t0 = time.time(); Xi, it, _ = solve_cgne(h); ti = time.time() - t0
        mem_ado = h.n_ado*h.d**2*16/1e6
        rows.append((Nel, h.d, "trunc" if trunc else "full", h.n_ado, td, mem_lu, ti, it, mem_ado,
                     abs(h.current(Xd, dops, 0) - h.current(Xi, dops, 0))))
print(f"{'Nel':>3} {'d':>3} {'hier.':>6} {'N_ADO':>6} | {'direct t[s]':>11} {'LU mem [MB]':>12} | {'CGNE t[s]':>10} {'iters':>6} {'ADO mem [MB]':>13} | {'|dI|':>8}")
for r in rows:
    print(f"{r[0]:3d} {r[1]:3d} {r[2]:>6} {r[3]:6d} | {r[4]:11.2f} {r[5]:12.2f} | {r[6]:10.2f} {r[7]:6d} {r[8]:13.3f} | {r[9]:8.1e}")

The memory column is the story: for the full hierarchy at $N_{\rm el}=3$ the sparse LU needs hundreds of MB (fill-in!) while the iterative solver needs well under a megabyte. Wall-clock time is a different matter — this is interpreted Python, and for tiny systems the direct solver wins. The paper's argument is about how cost *scales*: each CGNE iteration costs $\mathcal O(N_{\rm ADO}\cdot\#\text{terms}\cdot d^3)$ with $d^3=2^{3N_{\rm el}}$, versus $2^{6N_{\rm el}}$ for dense LU.

Finally, a system the direct solver cannot hold on a small machine (in my sandbox the sparse direct solve for the truncated 4-site problem was killed, most likely for running out of memory), solved iteratively:

In [ ]:
h, dops, land = make_heom(4, 2.0, truncate=True)
t0 = time.time(); X, it, hist = solve_cgne(h); dt = time.time() - t0
print(f"N_el=4 (d=16), N_ADO={h.n_ado}: CGNE converged in {it} iterations, {dt:.0f} s, "
      f"ADO storage {h.n_ado*h.d**2*16/1e6:.2f} MB")
print(f"I_HEOM = {h.current(X, dops, 0):.8f}   I_Landauer = {land:.8f}")

## 7. Take-aways and where to go next

* Casting the HEOM as **coupled matrix equations** lets a Krylov solver work with $d\times d$ matrices only; memory is just the ADOs themselves.
* In the eigenbasis of $H_S$ both the **preconditioner** and the **hierarchy truncation** reduce to element-wise divisions (diagonal Sylvester equations).
* In the toy model above, truncation gave the biggest and most reliable gain; preconditioning gave a modest, side-dependent one. Always measure on *your* system.

**Exercises.**
1. Add a Coulomb term $U\,n_1n_2$ to `chain_system` (the eigenbasis takes care of it). The hierarchy no longer closes at tier 2; set `NT=3` and compare direct vs. iterative solutions — and watch the memory.
2. Exploit the Hermiticity $\tilde\rho_{\bar j\dots}=\tilde\rho_{j\dots}^\dagger$ to halve the number of stored ADOs.
3. Lower the temperature (`T=0.2`) and increase `Np`; how do iteration counts change?
4. Replace the Python loop over terms by batched `einsum`/GPU operations — the paper's conclusion points to GPUs because the method is so memory-lean.

**Reference.** C. Kaspar and M. Thoss, *Efficient Steady-State Solver for the Hierarchical Equations of Motion Approach: Formulation and Application to Charge Transport through Nanosystems*, J. Phys. Chem. A **125**, 5190–5200 (2021), https://doi.org/10.1021/acs.jpca.1c02863